In [1]:
# ============================================================
# CUSTOMER CHURN INTELLIGENCE
# 03 - FEATURE ENGINEERING
# ============================================================

# %% [markdown]
# # Customer Churn Intelligence
# ## 03. Feature Engineering
#
# Objective:
# - Prepare customer data for machine learning
# - Create meaningful derived features
# - Encode categorical variables
# - Separate features and target
# - Split data into training and testing sets

# %%
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

# %% [markdown]
# ## 1. Load Clean Dataset

# %%
file_path = "../data/processed/telco_customer_churn_clean.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

display(df.head())

# %% [markdown]
# ## 2. Check Dataset

# %%
df.info()

# %%
print("Missing values:")
display(df.isnull().sum())

# %% [markdown]
# ## 3. Create Feature Engineering Copy

# %%
model_df = df.copy()

# %% [markdown]
# ## 4. Remove Customer ID
#
# customerID is an identifier and should not be used as a
# predictive feature.

# %%
model_df = model_df.drop(
    columns=["customerID"]
)

print("customerID removed.")

# %% [markdown]
# ## 5. Create Tenure-Based Features

# %%
model_df["TenureYears"] = (
    model_df["tenure"] / 12
)

model_df["IsNewCustomer"] = (
    model_df["tenure"] <= 6
).astype(int)

model_df["IsLongTermCustomer"] = (
    model_df["tenure"] >= 48
).astype(int)

# %%
display(
    model_df[
        [
            "tenure",
            "TenureYears",
            "IsNewCustomer",
            "IsLongTermCustomer"
        ]
    ].head()
)

# %% [markdown]
# ## 6. Create Revenue Features

# %%
model_df["EstimatedAnnualCharges"] = (
    model_df["MonthlyCharges"] * 12
)

model_df["ChargePerTenureMonth"] = (
    model_df["TotalCharges"] /
    model_df["tenure"].replace(0, np.nan)
)

model_df["ChargePerTenureMonth"] = (
    model_df["ChargePerTenureMonth"]
    .fillna(model_df["MonthlyCharges"])
)

# %%
display(
    model_df[
        [
            "MonthlyCharges",
            "TotalCharges",
            "EstimatedAnnualCharges",
            "ChargePerTenureMonth"
        ]
    ].head()
)

# %% [markdown]
# ## 7. Count Additional Services

# %%
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

# Convert service availability to a binary indicator.
#
# "Yes" = 1
# "No" = 0
# "No internet service" = 0

for column in service_columns:

    model_df[column + "_Flag"] = (
        model_df[column] == "Yes"
    ).astype(int)

# %%
model_df["TotalAdditionalServices"] = (
    model_df[
        [column + "_Flag" for column in service_columns]
    ].sum(axis=1)
)

# %%
display(
    model_df[
        [
            "TotalAdditionalServices"
        ]
    ].value_counts().sort_index()
)

# %% [markdown]
# ## 8. Create Family Feature

# %%
model_df["HasFamily"] = (
    (model_df["Partner"] == 1) |
    (model_df["Dependents"] == 1)
).astype(int)

# %% [markdown]
# ## 9. Create High Monthly Charge Indicator

# %%
monthly_charge_median = model_df["MonthlyCharges"].median()

model_df["HighMonthlyCharge"] = (
    model_df["MonthlyCharges"] >
    monthly_charge_median
).astype(int)

print(
    "Monthly charge median:",
    monthly_charge_median
)

# %% [markdown]
# ## 10. Create Contract Risk Feature

# %%
model_df["MonthToMonth"] = (
    model_df["Contract"] == "Month-to-month"
).astype(int)

# %% [markdown]
# ## 11. Create Payment Risk Feature

# %%
model_df["ElectronicCheck"] = (
    model_df["PaymentMethod"]
    == "Electronic check"
).astype(int)

# %% [markdown]
# ## 12. Inspect Engineered Features

# %%
engineered_columns = [
    "TenureYears",
    "IsNewCustomer",
    "IsLongTermCustomer",
    "EstimatedAnnualCharges",
    "ChargePerTenureMonth",
    "TotalAdditionalServices",
    "HasFamily",
    "HighMonthlyCharge",
    "MonthToMonth",
    "ElectronicCheck"
]

display(
    model_df[engineered_columns].head()
)

# %% [markdown]
# ## 13. Separate Target Variable

# %%
X = model_df.drop(
    columns=["Churn"]
)

y = model_df["Churn"]

print("Feature shape:", X.shape)
print("Target shape :", y.shape)

# %% [markdown]
# ## 14. Identify Categorical Columns

# %%
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

# %% [markdown]
# ## 15. Identify Numerical Columns

# %%
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

# %% [markdown]
# ## 16. One-Hot Encode Categorical Features

# %%
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Encoded feature shape:", X_encoded.shape)

display(X_encoded.head())

# %% [markdown]
# ## 17. Check for Missing Values

# %%
print(
    "Total missing values:",
    X_encoded.isnull().sum().sum()
)

# %% [markdown]
# ## 18. Train-Test Split
#
# We use stratification because Churn is an imbalanced target.

# %%
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

# %% [markdown]
# ## 19. Check Target Distribution

# %%
print("Training target distribution:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).mul(100).round(2))

# %% [markdown]
# ## 20. Scale Numerical Features
#
# Scaling is useful for models such as Logistic Regression.
#
# IMPORTANT:
# The scaler is fitted only on training data to prevent
# data leakage.

# %%
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Only scale original continuous numerical features
scale_columns = [
    column
    for column in [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "TenureYears",
        "EstimatedAnnualCharges",
        "ChargePerTenureMonth"
    ]
    if column in X_train.columns
]

X_train_scaled[scale_columns] = scaler.fit_transform(
    X_train[scale_columns]
)

X_test_scaled[scale_columns] = scaler.transform(
    X_test[scale_columns]
)

print("Scaled columns:")
print(scale_columns)

# %% [markdown]
# ## 21. Final Feature Summary

# %%
feature_summary = pd.DataFrame({
    "Feature": X_encoded.columns,
    "DataType": X_encoded.dtypes.astype(str).values
})

print(
    "Total ML features:",
    len(feature_summary)
)

display(feature_summary)

# %% [markdown]
# ## 22. Save ML Dataset

# %%
train_data = X_train_scaled.copy()
train_data["Churn"] = y_train.values

test_data = X_test_scaled.copy()
test_data["Churn"] = y_test.values

train_data.to_csv(
    "../data/processed/train.csv",
    index=False
)

test_data.to_csv(
    "../data/processed/test.csv",
    index=False
)

print("Training and testing datasets saved.")

# %% [markdown]
# ## 23. Save Feature Names

# %%
feature_names = pd.DataFrame({
    "Feature": X_encoded.columns
})

feature_names.to_csv(
    "../data/processed/feature_names.csv",
    index=False
)

print("Feature names saved.")

# %% [markdown]
# ## 24. Final Verification

# %%
print("Training shape:", train_data.shape)
print("Testing shape :", test_data.shape)

print("\nTraining preview:")
display(train_data.head())

# %% [markdown]
# # Feature Engineering Completed
#
# The data is now ready for machine learning.
#
# Next:
# - Logistic Regression
# - Random Forest
# - Gradient Boosting
# - Model comparison
# - Precision / Recall / F1
# - ROC-AUC
# - Confusion Matrix
# - Feature importance

Libraries imported successfully.
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   int64  
 4   Dependents        7043 non-null   int64  
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   int64  
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   int64  
 17  Paymen

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

customerID removed.


...

Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Scaled columns:
['tenure', 'MonthlyCharges', 'TotalCharges', 'TenureYears', 'EstimatedAnnualCharges', 'ChargePerTenureMonth']
Total ML features: 46


,Feature,DataType
0,SeniorCitizen,int64
1,Partner,int64
2,Dependents,int64
3,tenure,int64
4,PhoneService,int64
5,PaperlessBilling,int64
6,MonthlyCharges,float64
7,TotalCharges,float64
8,TenureYears,float64
9,IsNewCustomer,int64


Training and testing datasets saved.
Feature names saved.
Training shape: (5634, 47)
Testing shape : (1409, 47)

Training preview:


,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,TenureYears,IsNewCustomer,IsLongTermCustomer,EstimatedAnnualCharges,ChargePerTenureMonth,OnlineSecurity_Flag,OnlineBackup_Flag,DeviceProtection_Flag,TechSupport_Flag,StreamingTV_Flag,StreamingMovies_Flag,TotalAdditionalServices,HasFamily,HighMonthlyCharge,MonthToMonth,ElectronicCheck,gender_Male,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
3738,0,0,0,0.102371,0,0,-0.521976,-0.262257,0.102371,0,0,-0.521976,-0.539986,0,0,1,0,1,1,3,0,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0
3151,0,1,1,-0.711743,1,0,0.337478,-0.503635,-0.711743,0,0,0.337478,0.391496,1,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4860,0,1,1,-0.793155,0,0,-0.809013,-0.749883,-0.793155,0,0,-0.809013,-0.646102,1,1,0,1,0,0,3,1,0,0,0,1,1,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
3867,0,1,0,-0.263980,1,1,0.284384,-0.172722,-0.263980,0,0,0.284384,0.276552,0,1,1,0,1,1,4,1,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,1,0,0,0
3810,0,1,1,-1.281624,1,0,-0.676279,-0.989374,-1.281624,1,0,-0.676279,-0.674608,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
